# DWBI Week 5 — The ETL Process

**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A** · Superior University

PakFreight's Week-2 merge rebuilt as a pipeline that reruns: 37 rows land, 35 append, 34 load — 578,650 kg and PKR 4,808,000, exactly. Every number below is computed from `data/source/`, `data/dims/` or `data/night2/` in the notebook itself; nothing is typed in as a constant except the assertion target it is checked against.

## T3 — Order the seventeen steps (20 marks)

The steps are listed alphabetically in the workbook; run order is a mechanism,
not a style. The governing order is **shape, then text, then type, then keys**,
and the cadence rule is: *does this step need information that only exists
before the append?* If it does, it runs per source; if it does not, it runs
once on the combined table.

In [1]:
# T3 run order, built as data so the self-checks are computed, not typed
import pandas as pd
steps = pd.DataFrame([
    (1, "Connect to all three sources", "Extract", "per source"),
    (2, "Land a faithful copy, unchanged", "Extract", "per source"),
    (3, "Log source, time and row count", "Extract", "per source"),
    (4, "Promote headers; drop junk rows and columns", "Transform", "per source"),
    (5, "Rename every column to the common contract", "Transform", "per source"),
    (6, "Trim and clean all text columns", "Transform", "per source"),
    (7, "Normalise status casing to Proper Case", "Transform", "per source"),
    (8, "Parse dates with an explicit locale", "Transform", "per source"),
    (9, "Convert rail tons to kilograms; set types", "Transform", "per source"),
    (10, "Add ModeKey — assigned by source, not read", "Transform", "per source"),
    (11, "Append the three queries into one table", "Transform", "once"),
    (12, "Remove duplicates, now that rows compare", "Transform", "once"),
    (13, "Conform names, look up keys, unmatched to -1", "Transform", "once"),
    (14, "Load the four independent dimensions", "Load", "once"),
    (15, "Load DimRoute (needs city keys)", "Load", "once"),
    (16, "Load FactShipment", "Load", "once"),
    (17, "Reconcile, or fail the run", "Load", "once"),
], columns=["order", "step", "stage", "cadence"])
print(steps.stage.value_counts().to_string())
print(steps.cadence.value_counts().to_string())
assert sorted(steps.order) == list(range(1, 18))
assert (steps.stage == "Extract").sum() == 3 and (steps.stage == "Transform").sum() == 10
assert (steps.stage == "Load").sum() == 4
assert (steps.cadence == "per source").sum() == 10 and (steps.cadence == "once").sum() == 7

stage
Transform    10
Load          4
Extract       3
cadence
per source    10
once           7


**Analysis.** All six workbook self-checks pass from the table itself: 17 run-order numbers used exactly once, 3 extract + 10 transform + 4 load, and 10 per-source + 7 once. The cut falls exactly after step 10 — every step up to and including Add ModeKey runs per source, because mode, like the tons conversion and the date locale, is information that only exists before the append.

![Run order](charts/t3_steps.png)

| Order | Step | Stage | Cadence | Defects cleared | Why it sits here |
|---|---|---|---|---|---|
| 1 | Connect to all three sources | Extract | per source | — | Nothing can be shaped before it is read; each file is connected on its own terms (CSV vs named Excel sheet). |
| 2 | Land a faithful copy, unchanged | Extract | per source | — | Extract corrects nothing: 37 rows land, blank line and TOTAL footer included, so there is an untouched copy to prove the source against. |
| 3 | Log source, time and row count | Extract | per source | — | The receipt is written while the landing is still in front of you; a log written later is a reconstruction, not evidence. |
| 4 | Promote headers; drop junk rows and columns | Transform | per source | 2, 8 | Shape first: rail's title rows must be skipped *before* headers are promoted, and road's junk rows are dropped in the only file that has them. |
| 5 | Rename every column to the common contract | Transform | per source | 1 | The seven-name contract has to exist before the append, or the append cannot combine anything (see constraint 1). |
| 6 | Trim and clean all text columns | Transform | per source | 5 | Text before type before keys: a lookup key with a stray space is a different value, and no later step can recover it. |
| 7 | Normalise status casing to Proper Case | Transform | per source | 6 | Casing runs after trimming because `Text.Proper` on an untrimmed value preserves the space that breaks the match; together they collapse 17 strings to 3. |
| 8 | Parse dates with an explicit locale | Transform | per source | 4 | Dates are typed per source because only marine's text dates are ambiguous; the locale (en-GB) is stated, never guessed. |
| 9 | Convert rail tons to kilograms; set types | Transform | per source | 3 | Units are converted per source and *before* the append, so check 2 can read the rail subtotal at 295,250 kg on its own (see constraint 3). |
| 10 | Add ModeKey — assigned by source, not read | Transform | per source | 10 | The last per-source step, because after the append nothing can tell you which file a row came from. This is defect 10, handled as derivation, not repair. |
| 11 | Append the three queries into one table | Transform | once | — | 35 rows: 12 + 10 + 13, after the two junk rows are gone. Appending earlier would combine unrenamed, untrimmed, unconverted tables. |
| 12 | Remove duplicates, now that rows compare | Transform | once | 9 | Dedup runs once, on the combined table, on ShipmentID, *after* trim and uppercase — "now that rows compare" is in the step's name for a reason (see constraint 2). |
| 13 | Conform names, look up keys, unmatched to −1 | Transform | once | 7 | Keys last: the alias lookup needs trimmed spellings, and the −1 branch needs the full combined table so the count is meaningful. |
| 14 | Load the four independent dimensions | Load | once | — | DimDate, DimCity, DimMode, DimStatus have no dependencies on each other, so they load first and together. |
| 15 | Load DimRoute (needs city keys) | Load | once | — | DimRoute is a *dependent* dimension: its keys are city keys, so it cannot load until DimCity exists. |
| 16 | Load FactShipment | Load | once | — | The fact loads last among the tables because every one of its keys must already exist. |
| 17 | Reconcile, or fail the run | Load | once | — | The gate, not an afterthought: if any of the eight checks fails, the run fails rather than publishing a reconciled-looking lie. |

### The three constraints — and what breaking each one costs, as a number

| Constraint | Before → after | Cost of breaking it | Why |
|---|---|---|---|
| 1. Rename before append | RenamedToContract (5) → Append (11) | A fact table of **20 columns** | Seven names each, only `Status` shared: the append keeps every column present in any one table and fills the rest with null, so two thirds of every row is empty and no weight column ever combines. No error, no warning. |
| 2. Trim before deduplicate | TrimmedTextColumns (6) → RemovedDuplicates (12) | A duplicate that **survives** the dedup step | A duplicate that differs by a trailing space is not a duplicate to a byte comparison. Night 2 proves it: dedup-before-trim loads **36 rows / 591,550 kg** instead of 35 / 584,950. |
| 3. Convert before append | ConvertedTonsToKG (9) → Append (11) | **283,695.25 kg** instead of 578,650 kg | 206,500 + 76,900 + 295.25: the rail tons were never multiplied by 1,000, so the total is short by 294,954.75 kg and looks exact to two decimal places. |

### The hard question: why is trim-before-dedup a constraint if dedup-before-trim still gives 34 on night 1?

Because night 1 is lucky, and a rule that only holds on lucky data is not a rule. The two
RD-505 rows on night 1 are **byte-identical** — same ID, same date, same cities, same weight,
same cost, same casing — so even a whole-row comparison catches them, and dedup-before-trim
happens to give 34. That tells you nothing about whether the dedup rule is correct; it tells you
the test data never exercised it. Night 2 exercises it: the second RD-508 differs from the first
by **one trailing space** in the status field, so before trimming the two rows are different rows
and both survive, and the load reports 36 rows and 591,550 kg. The constraint holds because a
duplicate is defined by its **business key** (ShipmentID), and a business key only compares
truthfully after trimming and uppercasing — the comparison has to be made on the values the
business means, not on the bytes the file happened to contain. A pipeline whose correctness
depends on its duplicates being byte-identical is a pipeline waiting for its first trailing
space, and trailing spaces are the commonest dirt in the whole census.